# 原地算子优化

## 1. 功能简介

Dynamo 的 Functionalize 流程通常会把作用于模型输入的原地（In-place）算子转换为“非原地算子 + `copy_` 算子”，以保持 FX 图的函数式语义。对于 KV Cache 等本身就需要原地更新的输入，这种转换可能引入额外的数据搬运。

`input_inplace_pass=True` 会在受支持的算子模式上执行逆向变换，将这类组合恢复为输入原地操作；`inplace_pass=True` 则主要处理模型中间节点中的非原地算子。两者名称相近，但优化对象不同。

当前版本只支持[官方文档](https://gitcode.com/Ascend/torchair/blob/master/docs/zh/npugraph_ex/basic/inplace_pass.md)列出的算子进行优化，例如 `npu_kv_rmsnorm_rope_cache_v2`、`npu_mla_prolog_v3` 和 `npu_add_rms_norm_v2`。

**图 1**  input_inplace_pass 优化前后对比

<img src="./images/input_inplace_pass.svg" width="850">

如图所示，优化前需要先生成临时 Tensor，再通过 `copy_` 写回输入；优化后，受支持的算子可直接更新输入内存，从而减少临时 Tensor 和数据搬运。

In [ ]:
import torch
import torch_npu

assert torch.npu.is_available(), "请在已安装 CANN 和 torch_npu 的昇腾环境中运行"
torch.manual_seed(0)
print("PyTorch:", torch.__version__)
print("torch_npu:", torch_npu.__version__)
print("Device:", torch.npu.get_device_name(0))
# 该模型仅用于演示 Pass 的配置入口，不代表真实 KV Cache 融合模式。
class CacheUpdate(torch.nn.Module):
    def forward(self, hidden, cache):
        # 该示例展示配置位置；真实 KV Cache 算子须使用目标版本支持的
        # torch_npu 原生算子，不能用普通 copy_ 代替验证 Pass 效果。
        return hidden + cache


# 将模型迁移到 NPU 后，再交给 npugraph_ex 进行整图编译。
model = CacheUpdate().npu()
compiled = torch.compile(
    model,
    backend="npugraph_ex",
    options={"input_inplace_pass": True, "inplace_pass": True},
    fullgraph=True,
    dynamic=False,
)
# 模型和输入需位于同一设备；固定 Shape 便于复用已捕获的 NPUGraph。
hidden = torch.randn(2, 8, dtype=torch.float16).npu()
cache = torch.randn(2, 8, dtype=torch.float16).npu()
# 首次调用会触发编译与 Capture，后续同规格输入调用进入 Replay。
out = compiled(hidden, cache)
print("output shape:", tuple(out.shape))


## 2. 验证方法

- 打开 [TorchAir Debug Dump（图编译 Debug 信息保存）](https://gitcode.com/Ascend/torchair/blob/master/docs/zh/npugraph_ex/dfx/debug_save.md)，对比优化前后的 FX 图；
- 使用 Ascend PyTorch Profiler 检查 `copy_`、数据搬运和内存占用；
- 使用 `torch.testing.assert_close` 与 Eager 结果进行精度对比；
- 比较多次 Replay 的稳定区间，不要把首次编译耗时计入纯执行时间。

当输入本身很大时，`clone_input=True` 可能抵消原地化收益，应与下一节的内存复用策略联合测试。

## 3. 课后练习

### 一、单选题

（1）【单选题】Dynamo 的 Functionalize 流程通常如何处理作用于模型输入的原地算子？
- A. 直接删除该算子
- B. 转换为“非原地算子 + `copy_` 算子”
- C. 自动切换到 CPU 执行
- D. 强制每次重新 Capture

（2）【单选题】`input_inplace_pass=True` 主要优化的对象是？
- A. 模型参数的持久化存储
- B. 模型输入相关的“非原地算子 + copy_”组合
- C. 所有 Python 控制流
- D. 所有模型输出

（3）【单选题】`inplace_pass=True` 与 `input_inplace_pass=True` 的主要区别是？
- A. 前者处理模型中间节点，后者处理模型输入相关的节点
- B. 前者只能用于 CPU，后者只能用于 NPU
- C. 前者只影响精度，后者只影响 Shape
- D. 两者完全等价

（4）【单选题】验证原地算子优化是否减少数据搬运，最直接的工具是？
- A. Python 代码格式化器
- B. Ascend PyTorch Profiler
- C. pip 安装日志
- D. git 提交历史

（5）【单选题】当前 `input_inplace_pass` 的使用前提是什么？
- A. 任意原地算子都能自动优化
- B. 目标 TorchAir/TorchNPU 版本支持相应的算子模式
- C. 只能在动态 Shape 场景使用
- D. 必须关闭精度校验

（6）【单选题】当 user_inputs 本身很大时，开启 `clone_input=True` 最需要关注的风险是？
- A. 关闭 FX Graph
- B. 额外输入拷贝导致显存不足
- C. 强制使用多张 NPU
- D. 输出一定被覆盖

### 二、多选题

（7）【多选题】验证 `input_inplace_pass` 优化效果时，可以采取哪些方法？
- A. 对比优化前后的 FX 图或 Debug Dump
- B. 用 Profiler 检查 `copy_`、数据搬运和内存占用
- C. 使用 `torch.testing.assert_close` 对比 Eager 结果
- D. 只比较首次编译耗时

（8）【多选题】关于输入原地化优化，正确的说法有哪些？
- A. 它仅对目标版本支持的算子模式生效
- B. 它可减少临时 Tensor 和写回输入时的搬运
- C. 需要在目标 NPU 环境中验证实际收益
- D. 普通 `copy_` 示例一定能够证明该 Pass 生效

（9）【多选题】关于两个原地化 Pass 的描述，正确的有哪些？
- A. `input_inplace_pass` 关注模型输入相关的原地更新
- B. `inplace_pass` 关注模型中间节点的原地化机会
- C. 两个 Pass 可以按场景组合开启并分别验证
- D. 开启任一 Pass 都会禁用 Dynamo

（10）【多选题】评估原地算子优化时，哪些做法是合理的？
- A. 在多次 Replay 的稳定区间内比较性能
- B. 控制输入、随机种子和精度阈值
- C. 同时检查数值正确性和显存变化
- D. 仅根据图中节点数量判断优化收益

**运行以下代码单元查看参考答案与解析。**


In [ ]:
import os
answer_path = "answer/03.03_answer.txt"
if os.path.exists(answer_path):
    with open(answer_path, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("答案文件未找到，请检查 answer 目录。")
